Set literal's as variables<br>
If a code block was done more then once then I created a function for it<br>
I added scripting to send each of the AI's the challenge question, then asked each of the AI's to rank the different challenge questions and to get a short explanation of why<br>
Asked each of the AI's to answer the first challenge question by OPENAI. Then ran a compare of the response by the different AI's on all AI's for a ranking and short explanation why.<br>
My plan was to do this final step with all AI's but started getting lost in the code. <br>
I think a possible better set up would be use an Array of Hashs for the different AI's question/response/challenge. <br>I do not think Cursor (or Notebook) would be the best option for that.<br>
I did not test with GROQ or with OLLAMA<br>

In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

In [ ]:
OPEN_AI_NAME = "openai"
OPEN_AI_MODEL = "gpt-4o-mini"
ANTHROPIC_NAME = "anthropic"
ANTHROPIC_MODEL = "claude-3-7-sonnet-latest"
GOOGLE_NAME = "google"
GOOGLE_MODEL = "gemini-1.5-flash"
DEEPSEEK_NAME = "deepseek"
DEEPSEEK_MODEL = "deepseek-chat"
GROQ_NAME = "groq"
GROQ_MODEL = "llama-3.3-70b-versatile"
OLLAMA_NAME = "ollama"
OLLAMA_MODEL = "llama3.3"

ROLE_USER = "user"
TYPE_QUESTION = "question"
TYPE_ANSWER = "answer"

challenge_question_2_ask = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
challenge_question_2_ask += "Answer only with the question, no explanation."

In [ ]:
def get_judge_question(calling_api, type, statement, all_answers):
    judge_question_2_ask = f"""{calling_api} you are judging a competition between {len(all_answers)} competitors.
Each model has been given this {type}:

{statement}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with a list of the best competitor number and name, second best competitor number and name, third best competitor number and name, etc.
At the end of the list, give a brief explanation of why you ranked the competitors the way you did.
Do not list the {type}.

Here are the responses from each competitor:

{all_answers}

Do not include markdown formatting or code blocks.
"""
    return judge_question_2_ask


In [ ]:
def create_chat (model_type, api_key, base_url):
    if api_key is None:
        print(f"API key for {model_type} is not set")
        return None

    model_type_using_url = ("deepseek", "google", "groq", "ollama")
    chat_instance = None

    if model_type == "openai":
        chat_instance = OpenAI(api_key=api_key)
    elif model_type in model_type_using_url:
        chat_instance = OpenAI(api_key=api_key, base_url=base_url)
    elif model_type == "anthropic":
        chat_instance = Anthropic(api_key=api_key)
    else:
        print(f"Model type {model_type} is not supported")

    return chat_instance

In [ ]:
def ask_chat (model_type, chat_instance, model, messages, token_limit):
    response_content = None
    if model_type == "anthropic":
        response = chat_instance.messages.create(
            model=model,
            messages=messages,
            max_tokens=token_limit
        )
        response_content = response.content[0].text
    else:
        response = chat_instance.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=token_limit
        )
        response_content = response.choices[0].message.content
    return response_content
   


In [ ]:
def group_ai_responses_as_string(*args):
    group_ai_responses = ""
    for index, answer in enumerate(args):
        group_ai_responses += f"# Response from competitor {index+1}\n\n"
        group_ai_responses += answer + "\n\n"
    
    ##print ("group_ai_responses:\n", group_ai_responses)
    return group_ai_responses


In [ ]:
def get_model_messages(role, messages):
    return [{"role": role, "content": messages}]


In [ ]:
challenge_question_2_msgs = get_model_messages(ROLE_USER, challenge_question_2_ask)

openai_instance = create_chat(OPEN_AI_NAME, api_key=openai_api_key, base_url=None)
response_openai_challenge = ask_chat(
    OPEN_AI_NAME, openai_instance, OPEN_AI_MODEL, challenge_question_2_msgs, 1000)
print("OpenAI response: ", response_openai_challenge)

anthropic_instance = create_chat(ANTHROPIC_NAME, api_key=anthropic_api_key, base_url=None)
response_anthropic_challenge = ask_chat(
    ANTHROPIC_NAME, anthropic_instance, ANTHROPIC_MODEL, challenge_question_2_msgs, 1000)
print("Anthropic response: ", response_anthropic_challenge)

deepseek_instance = create_chat(DEEPSEEK_NAME, api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
response_deepseek_challenge = ask_chat(
    DEEPSEEK_NAME, deepseek_instance, DEEPSEEK_MODEL, challenge_question_2_msgs, 1000)
print("DeepSeek response: ", response_deepseek_challenge)

google_instance = create_chat(GOOGLE_NAME, api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response_google_challenge = ask_chat(
    GOOGLE_NAME, google_instance, GOOGLE_MODEL, challenge_question_2_msgs, 1000)
print("Google response: ", response_google_challenge)



In [ ]:
my_answers = group_ai_responses_as_string(
    f"{OPEN_AI_NAME}: {response_openai_challenge}", 
    f"{ANTHROPIC_NAME}: {response_anthropic_challenge}", 
    f"{DEEPSEEK_NAME}: {response_deepseek_challenge}", 
    f"{GOOGLE_NAME}: {response_google_challenge}")
###print ("my_answers:\n", my_answers)

compare_challenge_question = get_judge_question(OPEN_AI_NAME, TYPE_QUESTION, challenge_question_2_ask, my_answers)

compare_question_2_msgs = get_model_messages(ROLE_USER, compare_challenge_question)

response_openai_challenge_compare = ask_chat(
    OPEN_AI_NAME, openai_instance, OPEN_AI_MODEL, compare_question_2_msgs, 1000)
print("OpenAI response:\n", response_openai_challenge_compare)

response_anthropic_challenge_compare = ask_chat(
    ANTHROPIC_NAME, anthropic_instance, ANTHROPIC_MODEL, compare_question_2_msgs, 1000)
print("\nAnthropic response:\n", response_anthropic_challenge_compare)

response_deepseek_challenge_compare = ask_chat(
    DEEPSEEK_NAME, deepseek_instance, DEEPSEEK_MODEL, compare_question_2_msgs, 1000)
print("\nDeepSeek response:\n", response_deepseek_challenge_compare)

response_google_challenge_compare = ask_chat(
    GOOGLE_NAME, google_instance, GOOGLE_MODEL, compare_question_2_msgs, 1000)
print("\nGoogle response:\n", response_google_challenge_compare)


In [ ]:
#my_answers = group_ai_responses_as_string(
#    f"{OPEN_AI_NAME}: {response_openai_challenge}", 
#    f"{ANTHROPIC_NAME}: {response_anthropic_challenge}", 
#    f"{DEEPSEEK_NAME}: {response_deepseek_challenge}", 
#    f"{GOOGLE_NAME}: {response_google_challenge}")

#now we need to ask each of the AI models the response to the challenge question and then do a compare


In [ ]:
## OPEN AI first
challenge_answer_2_msgs = get_model_messages(ROLE_USER, response_openai_challenge)
##print ("OPEN AI challenge answer messages: ", response_openai_challenge)

response_openai_challenge_answer = ask_chat(
    OPEN_AI_NAME, openai_instance, OPEN_AI_MODEL, challenge_answer_2_msgs, 1000)
print("OpenAI response: ", response_openai_challenge_answer)

response_anthropic_challenge_answer = ask_chat(
    ANTHROPIC_NAME, anthropic_instance, ANTHROPIC_MODEL, challenge_answer_2_msgs, 1000)
print("\nAnthropic response:\n", response_anthropic_challenge_answer)

response_deepseek_challenge_answer = ask_chat(
    DEEPSEEK_NAME, deepseek_instance, DEEPSEEK_MODEL, challenge_answer_2_msgs, 1000)
print("\nDeepSeek response:\n", response_deepseek_challenge_answer)

response_google_challenge_answer = ask_chat(
    GOOGLE_NAME, google_instance, GOOGLE_MODEL, challenge_answer_2_msgs, 1000)
print("\nGoogle response:\n", response_google_challenge_answer)



In [ ]:
my_answers = group_ai_responses_as_string(
    f"{OPEN_AI_NAME}: {response_openai_challenge_answer}", 
    f"{ANTHROPIC_NAME}: {response_anthropic_challenge_answer}", 
    f"{DEEPSEEK_NAME}: {response_deepseek_challenge_answer}", 
    f"{GOOGLE_NAME}: {response_google_challenge_answer}")
###print ("my_answers:\n", my_answers)

compare_challenge_question_answer = get_judge_question(OPEN_AI_NAME, TYPE_ANSWER, challenge_question_2_ask, my_answers)

compare_question_answer_2_msgs = get_model_messages(ROLE_USER, compare_challenge_question_answer)

response_openai_question_answer_compare = ask_chat(
    OPEN_AI_NAME, openai_instance, OPEN_AI_MODEL, compare_question_answer_2_msgs, 1000)
print("OpenAI response:\n", response_openai_question_answer_compare)

response_anthropic_question_answer_compare = ask_chat(
    ANTHROPIC_NAME, anthropic_instance, ANTHROPIC_MODEL, compare_question_answer_2_msgs, 1000)
print("\nAnthropic response:\n", response_anthropic_question_answer_compare)

response_deepseek_question_answer_compare = ask_chat(
    DEEPSEEK_NAME, deepseek_instance, DEEPSEEK_MODEL, compare_question_answer_2_msgs, 1000)
print("\nDeepSeek response:\n", response_deepseek_question_answer_compare)

response_google_question_answer_compare = ask_chat(
    GOOGLE_NAME, google_instance, GOOGLE_MODEL, compare_question_answer_2_msgs, 1000)
print("\nGoogle response:\n", response_google_question_answer_compare)

In [ ]:
# decided to stop here. Would need to duplicate the code for each of the different models. And starting to get lost
# in the code. 
